In [1]:
from chggen.common.sample_utils import CSP_Generator
from chggen.common.data_utils import mkdir
from types import SimpleNamespace
import numpy as np

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csp = CSP_Generator(chggen_path = "./files/cut_7_conv_3_epoch=27-val_loss=0.87.ckpt",
                    device='cuda:6')

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


CHGNet initialized with 412,525 parameters
CHGNet will run on cuda:6


In [3]:
ld_kwargs = SimpleNamespace(
        n_step_each = 5,            # Corrector
        min_sigma = 0.01,
        num_noise_level = 200,
        signal_to_noise_ratio = 0.4,
        save_traj = False,
        disable_bar = False,
    )

In [4]:
gen_kwargs = SimpleNamespace(
        num_gen = 3, # number of structures generated from the cubic lattice
        num_mutation = 2, # number of mutations during the relax-generation iteration
        num_cell = 1, # number of times to the formula
        ehull_cutoff = 0.06,
        )

volume = 8


#  Generate a simple cubic structure via diffusion
s_list_cubic = csp.generate_simple_cubic_structure( comp_str= 'LiF', atom_volume=volume,
                                                   gen_kwargs = gen_kwargs, ld_kwargs = ld_kwargs)
                                                   

tensor([[2.5198, 2.5198, 2.5198],
        [2.5198, 2.5198, 2.5198],
        [2.5198, 2.5198, 2.5198]], device='cuda:6')


100%|██████████| 199/199 [00:18<00:00, 10.89it/s]


In [5]:
#  Generate seven different bravis lattices via diffusion
s_list_Bravis = csp.generate_structures_from_Bravis( comp_str= 'LiF', atom_volume=volume,
                                                    gen_kwargs=gen_kwargs, ld_kwargs=ld_kwargs)

tensor([[2.5198, 2.5198, 2.5198],
        [3.2660, 3.2660, 4.8990],
        [2.2013, 3.3019, 2.2013],
        [2.2132, 3.3197, 2.2132],
        [2.5517, 2.5517, 2.5517],
        [3.5095, 3.5095, 5.2643],
        [2.8286, 2.8286, 2.8286]], device='cuda:6')
tensor([[ 90.,  90.,  90.],
        [ 90.,  90.,  90.],
        [ 90.,  90.,  90.],
        [ 90., 100.,  90.],
        [ 80.,  85., 100.],
        [ 90.,  90., 120.],
        [ 60.,  60.,  60.]], device='cuda:6')


100%|██████████| 199/199 [00:15<00:00, 12.55it/s]


In [6]:
mkdir('files/volume_'+ str(volume))

for ii, s in enumerate(s_list_Bravis):
    s.to(filename='files/volume_'+ str(volume)+'/LiF_'+str(ii)+'.cif')

Folder exists


In [8]:
from chggen.common.sample_utils import get_inpaint_data_fromHost
from chggen.common.sample_utils import get_batch_inpaint_data_fromHost
from chggen.common.sample_utils import get_coarse_grain_framework



In [9]:
for s in s_list_cubic:
    s.remove_species(['Li'])

In [10]:
# s_list_cubic

In [11]:
gen_inputs_batch = get_batch_inpaint_data_fromHost(model=csp.chggen, 
                                       host_structure_list= s_list_cubic,
                                       num_intercalant_list= [1, 1, 1],
                                       ion = 'Li',
                                    )

/home/zhongpc/chggen/cond_gen/chggen/common/sample_utils.py:378: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  cur_frac_coords = torch.tensor(cur_frac_coords, dtype = torch.float32)


In [12]:
gen_inputs_batch

(tensor([9, 3, 9, 3, 9, 3], device='cuda:6', dtype=torch.int32),
 tensor([False,  True, False,  True, False,  True], device='cuda:6'),
 tensor([[0.9900, 0.6248, 0.0037],
         [0.2474, 0.2921, 0.3632],
         [0.0135, 0.9974, 0.9996],
         [0.0570, 0.4259, 0.2570],
         [0.9444, 0.8444, 0.8354],
         [0.9555, 0.9156, 0.2972]], device='cuda:6'),
 tensor([2, 2, 2], device='cuda:6', dtype=torch.int32),
 tensor([[90.0000, 90.0000, 90.0000],
         [90.0000, 90.0000, 90.0000],
         [90.0000, 90.0000, 90.0000]], device='cuda:6'),
 tensor([[2.5198, 2.5198, 2.5198],
         [2.5198, 2.5198, 2.5198],
         [2.5198, 2.5198, 2.5198]], device='cuda:6'))

In [13]:
s_inpaint_list= csp.generate_from_host_structure(host_structure_list= s_list_cubic,
                                 num_intercalant_list= [1, 1, 1],
                                 ld_kwargs=ld_kwargs, 
                                 species= 'Li')

100%|██████████| 959/959 [00:46<00:00, 20.61it/s]


In [14]:
mkdir('files/inpaint_volume_'+ str(volume))

for ii, s in enumerate(s_inpaint_list):
    s.to(filename='files/inpaint_volume_'+ str(volume)+'/LiF_'+str(ii)+'.cif')

Folder exists


In [15]:
s_CG_frame, symbol_CG, num_species = get_coarse_grain_framework(s_list_Bravis[-1], species_to_remove = 'Li')

symmetry:  0.1 angle_tolerance:  10
CG spacegroup:  Fm-3m


In [21]:
# Generate inpainted structures from the coarse-grained host structures
s_inpaint_list= csp.generate_from_host_structure(host_structure_list= [s_CG_frame]*5,
                                 num_intercalant_list= [1]*5,
                                 ld_kwargs=ld_kwargs, 
                                 species= 'Li')

/home/zhongpc/chggen/cond_gen/chggen/common/sample_utils.py:378: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  cur_frac_coords = torch.tensor(cur_frac_coords, dtype = torch.float32)
100%|██████████| 959/959 [00:47<00:00, 20.25it/s]


In [22]:
ROOT = 'files/volume_'+ str(np.round(s_CG_frame.volume / 2, 2))
mkdir(ROOT)

for ii, s in enumerate(s_inpaint_list):
    s.to(filename=ROOT +'/inpaint_LiF_'+str(ii)+'.cif')

Folder exists


64.00966618963801